In [1]:
def haptic_effect(
    action,
    difficulty,
    cognitive_effort,
    performance
):
    """
    Returns the adjustment to the learner-response logit
    produced by the chosen haptic action.

    action:
        0 = Low
        1 = Medium
        2 = High
    """

    need = (
        0.4 * difficulty
        + 0.4 * cognitive_effort
        + 0.2 * (1 - performance)
    )

    if action == 0:
        # Low feedback:
        # suitable when support need is low,
        # but insufficient when need is high
        effect = 0.05 - 0.30 * need

    elif action == 1:
        # Medium feedback:
        # broadly useful under moderate/high support need
        effect = 0.10 + 0.35 * need

    elif action == 2:
        # High feedback:
        # can help when support need is high,
        # but may be excessive when need is low
        effect = -0.15 + 0.75 * need

    else:
        raise ValueError("Action must be 0, 1, or 2")

    return effect

In [3]:
difficulty = 0.7
cognitive_effort = 0.6
performance = 0.4

need = (
    0.4 * difficulty
    + 0.4 * cognitive_effort
    + 0.2 * (1 - performance)
)

print("need =", need)

need = 0.64


In [7]:
import pandas as pd

df = pd.read_csv(
    "/Users/nataliakolbina/PycharmProjects/JupyterProject/Reproduce ReadMe+++/clear_test_readabert_gaze.csv"
)

state_df = df[
    [
        "Excerpt",
        "BT.Easiness",
        "Predicted_BT_Easiness",
        "Difficulty_D",
        "Gaze_Score_Raw"
    ]
].copy()

state_df["Cognitive_Effort_C"] = (
    state_df["Gaze_Score_Raw"] - state_df["Gaze_Score_Raw"].min()
) / (
    state_df["Gaze_Score_Raw"].max() - state_df["Gaze_Score_Raw"].min()
)

print(state_df.head())

                                             Excerpt  BT.Easiness  \
0  And outside before the palace a great garden w...    -1.054013   
1  One day he had gone beyond any point which he ...    -1.238432   
2  So off went Lionel to be made a King. He had n...    -0.188186   
3  Father had been away in the country for three ...    -1.009999   
4  It was very dark at first, perhaps because the...    -1.258699   

   Predicted_BT_Easiness  Difficulty_D  Gaze_Score_Raw  Cognitive_Effort_C  
0              -1.359378      0.570801       -7.549846            0.554339  
1              -0.568831      0.423853      -28.650097            0.325778  
2               0.189702      0.282855      -37.708939            0.227651  
3              -0.477061      0.406794      -22.028875            0.397500  
4              -0.832179      0.472804      -21.697756            0.401087  


In [8]:
state_df["need"] = (
    0.4 * state_df["Difficulty_D"]
    + 0.4 * state_df["Cognitive_Effort_C"]
    + 0.2 * (1 - state_df["Predicted_BT_Easiness"])
)

print(state_df[[
    "Difficulty_D",
    "Cognitive_Effort_C",
    "Predicted_BT_Easiness",
    "need"
]].head())

   Difficulty_D  Cognitive_Effort_C  Predicted_BT_Easiness      need
0      0.570801            0.554339              -1.359378  0.921932
1      0.423853            0.325778              -0.568831  0.613618
2      0.282855            0.227651               0.189702  0.366262
3      0.406794            0.397500              -0.477061  0.617130
4      0.472804            0.401087              -0.832179  0.715992


In [9]:
def haptic_effect(
    action,
    difficulty,
    cognitive_effort,
    performance
):
    """
    Returns the adjustment to the learner-response logit
    produced by the chosen haptic action.

    action:
        0 = Low
        1 = Medium
        2 = High
    """

    need = (
        0.4 * difficulty
        + 0.4 * cognitive_effort
        + 0.2 * (1 - performance)
    )

    if action == 0:


        # suitable when support need is low,
        # but insufficient when need is high
        effect = 0.05 - 0.30 * need

    elif action == 1:
        # Medium feedback:
        # broadly useful under moderate/high support need
        effect = 0.10 + 0.35 * need

    elif action == 2:
        # High feedback:
        # can help when support need is high,
        # but may be excessive when need is low
        effect = -0.15 + 0.75 * need

    else:
        raise ValueError("Action must be 0, 1, or 2")

    return effect

In [10]:
need = (
    0.4 * difficulty
    + 0.4 * cognitive_effort
    + 0.2 * (1 - performance)
)

In [11]:
def probability_correct_with_haptics(
    ability,
    difficulty,
    cognitive_effort,
    performance,
    action,
    alpha=4.0,
    beta=3.0,
    gamma=2.0
):
    base_logit = (
        alpha * ability
        - beta * difficulty
        - gamma * cognitive_effort
    )

    effect = haptic_effect(
        action=action,
        difficulty=difficulty,
        cognitive_effort=cognitive_effort,
        performance=performance
    )

    final_logit = base_logit + effect

    return 1 / (1 + np.exp(-final_logit))

In [14]:
import math

HAPTIC_ACTIONS = {
    0: "none",
    1: "moderate",
    2: "strong"
}

def haptic_effect(action, need):
    if action == 0:
        return 0.0
    elif action == 1:
        return 0.15 * need
    elif action == 2:
        return 0.30 * need
    else:
        raise ValueError("action must be 0, 1, or 2")

def probability_correct_with_haptics(
    ability,
    difficulty,
    cognitive_effort,
    performance,
    action,
    alpha=2.0,
    beta=2.0,
    gamma=1.5
):
    need = (
        0.4 * difficulty
        + 0.4 * cognitive_effort
        + 0.2 * (1 - performance)
    )

    effect = haptic_effect(action, need)

    score = (
        alpha * ability
        - beta * difficulty
        - gamma * cognitive_effort
        + effect
    )

    p_correct = 1 / (1 + math.exp(-score))
    return p_correct


In [15]:
test_cases = [
    {
        "name": "Strong learner / low need",
        "ability": 0.80,
        "D": 0.20,
        "C": 0.20,
        "P": 0.85
    },
    {
        "name": "Average learner / medium need",
        "ability": 0.55,
        "D": 0.50,
        "C": 0.50,
        "P": 0.50
    },
    {
        "name": "Struggling learner / high need",
        "ability": 0.30,
        "D": 0.80,
        "C": 0.80,
        "P": 0.20
    }
]

for case in test_cases:
    print("\n", case["name"])

    for action in [0, 1, 2]:
        p = probability_correct_with_haptics(
            ability=case["ability"],
            difficulty=case["D"],
            cognitive_effort=case["C"],
            performance=case["P"],
            action=action
        )

        print(
            HAPTIC_ACTIONS[action],
            "P(correct) =",
            round(p, 3)
        )


 Strong learner / low need
none P(correct) = 0.711
moderate P(correct) = 0.717
strong P(correct) = 0.723

 Average learner / medium need
none P(correct) = 0.343
moderate P(correct) = 0.36
strong P(correct) = 0.378

 Struggling learner / high need
none P(correct) = 0.1
moderate P(correct) = 0.111
strong P(correct) = 0.123


In [16]:
for need_case in [
    (0.2, 0.2, 0.9),
    (0.5, 0.5, 0.5),
    (0.8, 0.8, 0.2)
]:
    D, C, P = need_case

    need = (
        0.4 * D
        + 0.4 * C
        + 0.2 * (1 - P)
    )

    print(
        "\nState:",
        f"D={D}, C={C}, P={P}, need={round(need, 3)}"
    )

    for action in [0, 1, 2]:
        effect = haptic_effect(action, need)

        print(
            HAPTIC_ACTIONS[action],
            "effect =",
            round(effect, 3)
        )


State: D=0.2, C=0.2, P=0.9, need=0.18
none effect = 0.0
moderate effect = 0.027
strong effect = 0.054

State: D=0.5, C=0.5, P=0.5, need=0.5
none effect = 0.0
moderate effect = 0.075
strong effect = 0.15

State: D=0.8, C=0.8, P=0.2, need=0.8
none effect = 0.0
moderate effect = 0.12
strong effect = 0.24


In [17]:
def haptic_effect(action, difficulty, cognitive_effort, performance):
    need = (
        0.4 * difficulty
        + 0.4 * cognitive_effort
        + 0.2 * (1 - performance)
    )

    if action == 0:
        return 0.0
    elif action == 1:
        return 0.15 * need
    elif action == 2:
        return 0.30 * need
    else:
        raise ValueError("action must be 0, 1, or 2")

In [18]:
haptic_cost = {
    0: 0.00,   # Low
    1: 0.03,   # Medium
    2: 0.08    # High
}